# Lab 5a: Adversarial Attack on a Malware Classifier

## Lab Objective:

In this lab, students will use a pre-trained CNN malware classifier, a sterilized Gatak sample image, and the Fast Gradient Sign Method (FGSM) to create an adversarial image.

There is no live malware in this lab. The sample is an inert PNG representation of bytecode texture.

## Acknowledgement

This module was developed under the National Science Foundation awards # 2416990, 2416992 and 2607393 at Tennessee Tech University, North Carolina A&T State University and University of Albany.  

## Setup

In [ ]:
# NOTHING TO DO HERE - JUST RUN CELL
# Install needed modules
!pip install kagglehub keras shap tensorflow tensorflow_datasets ipywidgets matplotlib scikit-learn

## Files Needed

Run this notebook from the lab directory, or upload these files when prompted in Colab:

- `MalwareClassifier.h5`
- `DsXzfyoVUlSAm8PwtZOB.bytes.png` from `gatak_only/Gatak/`

In [ ]:
# Runtime setup
import os
from pathlib import Path

# Keep TensorFlow logs quieter in notebook output.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

LABELS = [
    "Gatak", "Kelihos_ver1", "Kelihos_ver3", "Lollipop",
    "Obfuscator_ACY", "Ramnit", "Tracur", "Vundo"
]

MODEL_PATH = Path("MalwareClassifier.h5")
SAMPLE_PATH = Path("gatak_only/Gatak/DsXzfyoVUlSAm8PwtZOB.bytes.png")

def running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def ensure_lab_files():
    """Find or upload the model and sample image needed by this notebook."""
    if MODEL_PATH.exists() and SAMPLE_PATH.exists():
        print("Found lab files in the current working directory.")
        return

    if not running_in_colab():
        missing = [str(p) for p in [MODEL_PATH, SAMPLE_PATH] if not p.exists()]
        raise FileNotFoundError(
            "Missing required lab file(s): " + ", ".join(missing) +
            ". Run this notebook from the lab directory or upload the files."
        )

    from google.colab import files
    print("Upload MalwareClassifier.h5 and DsXzfyoVUlSAm8PwtZOB.bytes.png when prompted.")
    uploaded = files.upload()

    if not MODEL_PATH.exists() and "MalwareClassifier.h5" in uploaded:
        Path("MalwareClassifier.h5").write_bytes(uploaded["MalwareClassifier.h5"])

    sample_name = "DsXzfyoVUlSAm8PwtZOB.bytes.png"
    if not SAMPLE_PATH.exists() and sample_name in uploaded:
        SAMPLE_PATH.parent.mkdir(parents=True, exist_ok=True)
        SAMPLE_PATH.write_bytes(uploaded[sample_name])

    if not MODEL_PATH.exists() or not SAMPLE_PATH.exists():
        raise FileNotFoundError("The model and sample PNG are both required before continuing.")

ensure_lab_files()
print("TensorFlow:", tf.__version__)
print("Keras:", getattr(keras, "__version__", "unknown"))


## Load the Model and Sample

The original script used `ImageDataGenerator().flow_from_directory(...)`. For Colab, this notebook loads the single PNG directly, resizes it to the model's expected `64 x 64 x 3` input, and normalizes pixel values to `0..1`.

In [ ]:
# Load the sample image and the pre-trained malware classifier.
def load_sample_image(path=SAMPLE_PATH):
    image = keras.utils.load_img(path, target_size=(64, 64), color_mode="rgb")
    array = keras.utils.img_to_array(image).astype("float32") / 255.0
    return np.expand_dims(array, axis=0)

x_original = load_sample_image()
y_true = np.zeros((1, len(LABELS)), dtype="float32")
y_true[0, LABELS.index("Gatak")] = 1.0

# compile=False avoids legacy optimizer warnings from the older .h5 export.
malware_model = keras.models.load_model(MODEL_PATH, compile=False)
malware_model.summary()


In [ ]:
# Prediction and plotting helpers.
def format_percent(value):
    return f"{value * 100:.6f}".rstrip("0").rstrip(".")

def prediction_table(probabilities):
    probabilities = np.asarray(probabilities).reshape(-1)
    rows = []
    for label, probability in zip(LABELS, probabilities):
        rows.append((label, probability, format_percent(probability)))
    return rows

def top_prediction(probabilities):
    probabilities = np.asarray(probabilities).reshape(-1)
    index = int(np.argmax(probabilities))
    return LABELS[index], float(probabilities[index])

def print_prediction(probabilities):
    label, confidence = top_prediction(probabilities)
    print(f"Prediction: {label} ({format_percent(confidence)}% confidence)")
    print()
    print("Class probabilities:")
    for label_name, probability, percent in prediction_table(probabilities):
        print(f"  {label_name:15s} {percent:>12s}%")

def show_image(image, title):
    plt.figure(figsize=(4, 4))
    plt.imshow(np.clip(image, 0, 1))
    plt.title(title)
    plt.axis("off")
    plt.show()


## Step 1: Classify the Original Gatak Sample

Run the cell below and record whether the model correctly classifies the unedited Gatak sample. Save or screenshot the displayed plot for your answer document.

In [ ]:
# Prediction on the unedited Gatak sample.
base_probabilities = malware_model.predict(x_original, verbose=0)[0]
print_prediction(base_probabilities)
base_label, base_confidence = top_prediction(base_probabilities)
show_image(x_original[0], f"Original sample\n{base_label}: {format_percent(base_confidence)}% confidence")


**Student response:**

- Did the model correctly classify the original Gatak sample?
- Screenshot or saved image: insert into your answer document.

## Step 2: Create the FGSM Adversarial Pattern

FGSM uses the gradient of the classification loss with respect to the input image. The sign of that gradient gives a direction that increases model error. The plot below visualizes that signed pattern.

In [ ]:
# FGSM adversarial pattern.
# This follows the TensorFlow FGSM tutorial approach: take the sign of the
# gradient of the loss with respect to the input image.
loss_object = tf.keras.losses.CategoricalCrossentropy()

def create_adversarial_pattern(input_image, input_label):
    input_image = tf.convert_to_tensor(input_image, dtype=tf.float32)
    input_label = tf.convert_to_tensor(input_label, dtype=tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(input_image)
        prediction = malware_model(input_image)
        loss = loss_object(input_label, prediction)

    gradient = tape.gradient(loss, input_image)
    signed_gradient = tf.sign(gradient)
    return signed_gradient.numpy()

perturbation = create_adversarial_pattern(x_original, y_true)
show_image(perturbation[0] * 0.5 + 0.5, "Adversarial pattern")


**Student response:**

- Screenshot or saved image of the adversarial pattern: insert into your answer document.

## Step 3: Create an Adversarial Image

The original script adds `epsilon * perturbation` to the base image. Start with `epsilon = 0.15`, then try smaller values. The best attack uses the smallest visible change that still disrupts the classifier.

In [ ]:
# Create an adversarial image.
# Try different epsilon values between 0 and 1. Two decimal places are enough
# for this lab. Smaller values are better if they still fool the model.
epsilon = 0.15

adv_image = x_original.copy()
adv_image[0] = adv_image[0] + epsilon * perturbation[0]

adv_probabilities = malware_model.predict(adv_image, verbose=0)[0]
print_prediction(adv_probabilities)
adv_label, adv_confidence = top_prediction(adv_probabilities)
show_image(
    adv_image[0],
    f"Adversarial image\nEpsilon: {epsilon}\n{adv_label}: {format_percent(adv_confidence)}% confidence",
)


**Student response:**

- Did the model classify the adversarial sample correctly?
- If not, what family did it predict and with what confidence?
- If you rerun the same epsilon, does it produce the same prediction and confidence?

## Fine-Tune Epsilon

Use the helper below or edit `epsilon` above. Values should be between `0` and `1`; two decimal places are sufficient for this lab.

In [ ]:
# Optional helper: test a range of epsilon values without editing and rerunning
# the earlier cell each time.
epsilon_values = [0.00, 0.01, 0.02, 0.05, 0.10, 0.15, 0.20]

for eps in epsilon_values:
    candidate = x_original.copy()
    candidate[0] = candidate[0] + eps * perturbation[0]
    probabilities = malware_model.predict(candidate, verbose=0)[0]
    label, confidence = top_prediction(probabilities)
    print(f"epsilon={eps:>4.2f} -> {label:15s} {format_percent(confidence):>12s}%")


**Student response:**

- Best epsilon value:
- Visible differences from the original image:
- Final predicted family and confidence:
- Screenshot or saved image of your best adversarial image: insert into your answer document.